# 2주차 · 영화 리뷰 감성 분석 (NLP) 🎬

이번 주 미니 대회는 **컴퓨터에게 글의 감정을 읽게 하기**! 네이버 영화 리뷰 텍스트를 보고 **긍정(1)인지 부정(0)인지** 맞히는 문제예요.

한 가지 함정이 있어요 — 긍정 리뷰가 전체의 약 25%뿐인 **불균형 데이터**입니다. 그래서 "다 부정!"이라고 찍으면 정확도는 75%처럼 보이지만, 평가 지표인 **F1 점수는 0점**이에요. F1은 소수 클래스(긍정)를 얼마나 잘 찾아내는지를 보는 지표거든요.

**진행 방법 (꼭 읽어주세요!)**
1. 셀을 **위에서부터 하나씩** `Shift+Enter`로 실행하세요.
2. **✏️ 표시된 셀만** 바꿔보면 됩니다.
3. 마지막 셀이 `submission.csv`를 다운로드해줘요 → 웹 **Submit 탭**에 업로드.

- 평가 지표: **F1** (높을수록 좋음, 0~1 사이)
- 기본 코드 그대로도 베이스라인은 이깁니다. 완주가 먼저! 😊

## 1. 데이터 불러오기  *(그냥 실행하세요)*

리뷰 데이터를 읽어옵니다. `document` 컬럼이 리뷰 텍스트, `target`이 정답(1=긍정, 0=부정)이에요. 텍스트가 비어있는(결측) 리뷰는 에러를 막기 위해 빈 문자열로 채웁니다.

실행하면 데이터 크기와 함께 `긍정 비율: 0.25` 정도가 찍혀요 — 이게 바로 아까 말한 불균형이에요!

In [5]:
import pandas as pd  # 표 데이터를 다루는 라이브러리

# 데이터 공개 링크 (운영진이 채워둔 것 — 수정하지 마세요!)
TRAIN_URL = "https://raw.githubusercontent.com/HUFS-DAT/dat-datasets/main/nlp/train.csv"
TEST_URL  = "https://raw.githubusercontent.com/HUFS-DAT/dat-datasets/main/nlp/test.csv"

if TRAIN_URL:
    train = pd.read_csv(TRAIN_URL); test = pd.read_csv(TEST_URL)  # URL에서 바로 읽기
else:
    # (예비용) URL이 없을 때만 직접 업로드 — 보통 실행되지 않아요
    from google.colab import files; files.upload()
    train = pd.read_csv("train.csv"); test = pd.read_csv("test.csv")

# 텍스트가 비어있는(NaN) 리뷰가 있으면 에러가 나므로 빈 문자열("")로 채웁니다
train["document"] = train["document"].fillna("")
test["document"] = test["document"].fillna("")

print("train:", train.shape, "| test:", test.shape)
print("긍정 비율:", round(train["target"].mean(), 2))  # 0.25 근처 = 불균형!
train.head(3)  # 리뷰 3개 미리보기

train: (70158, 3) | test: (30068, 2)
긍정 비율: 0.25


,id,document,target
0,9661597,어떤 의미에선 정말로 공포스럽다. (부들부들~~),0
1,9394829,이야기 전개가 느린데다가 결말이 진부하군,0
2,6169222,어찌 만화가 더 무섭다냐...,0


## 2. ✏️ 여기를 바꿔보세요 — 텍스트를 숫자로 바꾸고 모델 학습

컴퓨터는 글자를 바로 이해하지 못해서, 먼저 **TF-IDF**라는 방법으로 텍스트를 숫자표로 바꿉니다. TF-IDF는 "이 리뷰에 어떤 단어가 얼마나 특징적으로 쓰였나"를 점수로 매기는 방법이에요. 그 숫자표를 **로지스틱 회귀**(가장 기본적인 분류 모델)에 넣어 긍정/부정을 배우게 합니다.

`class_weight='balanced'`가 핵심 — 수가 적은 긍정 리뷰를 틀리면 벌점을 더 크게 줘서, 불균형 데이터에서도 긍정을 놓치지 않게 해줘요.

실행하면 몇십 초 안에 끝나고, 마지막에 모델 정보가 출력되면 성공이에요.

**점수(F1) 올리는 실험 아이디어** — 하나씩 바꿔서 제출해보세요:
- `ngram_range=(1, 2)` → `(1, 3)`: 단어 1개·2개짜리 조합에 더해 3개짜리 표현("정말 재미 없다")까지 보기
- `max_features=50000` → `100000`: 기억하는 단어 수 늘리기
- `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))`: 단어 대신 **글자 조각** 단위로 보기 — 한국어처럼 띄어쓰기가 들쭉날쭉한 텍스트에 효과적일 때가 많아요
- 더 도전: `konlpy` 형태소 분석, 사전학습 모델(BERT 계열)

In [6]:
# 설치 및 환경
!pip install -q transformers datasets evaluate

import torch
print("GPU 사용 가능:", torch.cuda.is_available())

GPU 사용 가능: True


In [15]:
print(train["document"].str.len().describe())

count    70158.000000
mean        35.582072
std         30.037238
min          1.000000
25%         16.000000
50%         27.000000
75%         43.000000
max        146.000000
Name: document, dtype: float64


In [16]:
# 검증 셋 분리
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    train['document'], train['target'],
    test_size=0.2, random_state=42, stratify=train['target']
)

MAX_LEN = 64

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
import pandas as pd
from datasets import Dataset

def to_hf_dataset(texts, labels=None):
    df = pd.DataFrame({"document": texts.reset_index(drop=True)})
    if labels is not None:
        df["label"] = labels.reset_index(drop=True)
    return Dataset.from_pandas(df)

def tokenize_fn(batch):
    return tokenizer(batch["document"], truncation=True, max_length=MAX_LEN)

train_ds = to_hf_dataset(X_tr, y_tr).map(tokenize_fn, batched=True)
val_ds   = to_hf_dataset(X_val, y_val).map(tokenize_fn, batched=True)

Map:   0%|          | 0/56126 [00:00<?, ? examples/s]

Map:   0%|          | 0/14032 [00:00<?, ? examples/s]

In [19]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from transformers import Trainer, DataCollatorWithPadding

class_weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weights = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # 동적 패딩 담당

In [20]:
from transformers import TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"f1": f1_score(labels, preds)}

args = TrainingArguments(
    output_dir="./bert_out",
    per_device_train_batch_size=32,     # 16 → 32 (짧은 max_length라 메모리 여유 생김)
    per_device_eval_batch_size=64,
    num_train_epochs=2,                 # 3 → 2 (일단 빠르게 결과 확보)
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=True,                          # 속도 약 2배 향상 (GPU 필수)
    dataloader_num_workers=2,           # 데이터 로딩 병목 완화
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=data_collator,        # 동적 패딩 적용
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.280052,0.269654,0.789157
2,0.179844,0.312424,0.819267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3508, training_loss=0.2512783675944112, metrics={'train_runtime': 451.824, 'train_samples_per_second': 248.442, 'train_steps_per_second': 7.764, 'total_flos': 3571508521003920.0, 'train_loss': 0.2512783675944112, 'epoch': 2.0})

In [21]:
# threshold tuning

val_logits = trainer.predict(val_ds).predictions
val_proba = torch.softmax(torch.tensor(val_logits), dim=1)[:, 1].numpy()

best_f1, best_t = 0, 0.5
for t in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y_val, (val_proba >= t).astype(int))
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"최적 threshold: {best_t:.2f}, 검증 F1: {best_f1:.4f}")

최적 threshold: 0.70, 검증 F1: 0.8276


## 3. 예측 & 제출 파일 저장  *(실행하면 자동 다운로드)*

학습된 모델로 test 리뷰들의 긍정/부정을 예측하고 `submission.csv`로 저장합니다. 실행하면 파일이 **자동으로 다운로드**돼요.

👉 다운로드된 `submission.csv`를 웹사이트의 **Submit 탭**에 업로드하면 F1 점수가 나옵니다. 여러 번 제출할 수 있으니 위 셀을 바꿔가며 점수를 올려보세요!

In [22]:
test_ds = to_hf_dataset(test["document"]).map(tokenize_fn, batched=True)

test_logits = trainer.predict(test_ds).predictions
test_proba = torch.softmax(torch.tensor(test_logits), dim=1)[:, 1].numpy()

test["prediction"] = (test_proba >= best_t).astype(int)
test[["id", "prediction"]].to_csv("submission.csv", index=False)
print("저장 완료")

try:
    # 코랩에서 실행 중이면 자동 다운로드
    from google.colab import files; files.download("submission.csv")
except Exception:
    print("왼쪽 파일탭에서 submission.csv를 내려받으세요.")

Map:   0%|          | 0/30068 [00:00<?, ? examples/s]

저장 완료


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🆘 막히면 여기를 보세요

| 증상 | 해결 |
|---|---|
| `NameError: name 'train' is not defined` | 위쪽 셀을 건너뛰었어요. 맨 위부터 순서대로 실행하세요. |
| `ValueError: np.nan is an invalid document` | 1번 셀의 `fillna("")` 부분이 실행 안 된 거예요. 1번 셀부터 다시 실행. |
| 학습이 1분 넘게 걸림 | `max_features`를 크게 키우면 오래 걸려요. 정상이니 기다리거나 값을 줄이세요. |
| 데이터 로드 URL 에러 | 인터넷 연결 확인 후 셀을 한 번 더 실행해보세요. |

텍스트도 결국 숫자로 바꾸면 머신러닝이 된다는 것, 오늘의 핵심이었어요! 🙌

## 더 나아가기 (선택)

점수는 파라미터 숫자보다 **"데이터를 어떻게 보여주느냐"**(피처 엔지니어링·검증 전략·앙상블)에서 갈립니다. 아래는 정답 코드가 아니라 **방향과 시작점 몇 줄**만 적어둔 것이에요.

> AI(ChatGPT·Claude·Gemini)에게 물어보며 해도 좋고, 코랩을 자유롭게 고치거나 새 셀을 만들어도 됩니다. **제출 형식(id, prediction)만 맞으면 무엇을 하든 괜찮아요.**

평가 지표는 **F1 (macro)** 이라, 수가 적은 긍정 리뷰를 얼마나 잘 잡느냐가 중요해요. 아래 갈래들을 실험해보세요.

- **TF-IDF 세밀하게**: `min_df`/`max_df`로 너무 드물거나 흔한 단어 걸러내기, `sublinear_tf=True`
- **글자 단위(char) ngram**: 띄어쓰기가 들쭉날쭉한 한국어에 효과적일 때가 많아요. 단어+글자를 함께 쓰는 FeatureUnion
- **다른 모델**: `LinearSVC`, `SGDClassifier`, `C` 값 튜닝, 여러 모델 앙상블
- **로컬 검증**: `cross_val_score`로 `f1_macro`를 제출 전에 미리 재보기
- **더 멀리**: konlpy 형태소 분석(설치가 무거움), 사전학습 한국어 모델(KoBERT 계열) — 선배들이 쓰는 갈래


In [ ]:
# ── TF-IDF 더 세밀하게 + 단어/글자 결합 ──
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.pipeline import FeatureUnion
# min_df=2 (드문 단어 무시), max_df=0.9 (흔한 단어 무시), sublinear_tf=True 를 실험해보세요.
# 단어 기반 + 글자 기반을 합치면 한국어에서 효과가 좋을 때가 많아요:
# word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
# char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=2)
# vec = FeatureUnion([("word", word), ("char", char)])
# Xtr = vec.fit_transform(train["document"]); Xte = vec.transform(test["document"])


In [ ]:
# ── 다른 모델 & 로컬 검증 (F1 macro) ──
# from sklearn.svm import LinearSVC
# from sklearn.linear_model import SGDClassifier
# from sklearn.model_selection import cross_val_score
# LinearSVC(C=1.0, class_weight="balanced") — 텍스트 분류에서 강력한 기본기
# C 값을 0.1, 0.3, 1, 3 등으로 바꿔 규제 세기를 조절해보세요.
# 제출 전 F1 macro를 미리 재보기 (평가 지표와 동일하게):
# print(cross_val_score(clf, Xtr, train["target"], cv=5, scoring="f1_macro").mean())


In [ ]:
# ── 더 멀리: 형태소 분석 · 사전학습 모델 ──
# 형태소 분석기로 토큰을 잘라 넣으면 성능이 오를 수 있어요 (설치가 조금 무겁습니다):
# !pip install konlpy
# from konlpy.tag import Okt
# okt = Okt()
# tokens = train["document"].map(lambda s: " ".join(okt.morphs(s)))
# 이 결과를 TfidfVectorizer에 넣으면 됩니다 (첫 실행은 시간이 걸려요).
# 선배들이 쓰는 갈래: 사전학습 한국어 모델(KoBERT 등) 미세조정 — 난이도는 높지만 상한이 큽니다.
